# Diabetes Detection Agent

Predicts diabetes risk two ways and lets an agent explain the results:

1. **Random Forest** trained on the Pima Indians Diabetes dataset
2. **Memory-based (ACT-R style)**: find similar past patients; if 60% or more were diabetic, predict diabetic

The agent checks the input, calls the tool, compares both results, and adds a disclaimer.

In [ ]:
import pandas as pd
from agents import Agent, Runner, function_tool
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

load_dotenv();

## 1. Load data

In [ ]:
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
        "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

df = pd.read_csv(URL, names=COLS)
print(df.shape)
df.head()

## 2. Random Forest

In [ ]:
X, y = df[COLS[:-1]], df["Outcome"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
print(f"Test accuracy: {rf.score(X_te, y_te):.0%}")

In [ ]:
pd.Series(rf.feature_importances_, index=COLS[:-1]).sort_values().plot.barh(title="Feature importance");

## 3. Memory-based prediction (ACT-R style)

A past patient is "similar" if glucose is within 15, BMI within 3, and age within 5. Fewer than 5 similar patients means low confidence.

In [ ]:
def memory_predict(glucose, bmi, age):
    m = df[(df.Glucose.sub(glucose).abs() <= 15) & (df.BMI.sub(bmi).abs() <= 3) & (df.Age.sub(age).abs() <= 5)]
    if m.empty:
        return {"matches": 0, "prediction": "unknown", "confidence": "none"}
    share = float(m.Outcome.mean())
    return {"matches": len(m), "diabetic_share": round(share, 2),
            "prediction": "diabetic" if share >= 0.6 else "not diabetic",
            "confidence": "low" if len(m) < 5 else "normal"}

memory_predict(148, 33.6, 50)

## 4. Tool and agent

In [ ]:
@function_tool
def assess_risk(pregnancies: int, glucose: float, blood_pressure: float, skin_thickness: float,
                insulin: float, bmi: float, pedigree: float, age: int) -> dict:
    """Run the Random Forest model and the memory-based model on one patient."""
    row = pd.DataFrame([[pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, pedigree, age]],
                       columns=COLS[:-1])
    p = float(rf.predict_proba(row)[0][1])
    return {"random_forest": {"diabetic_probability": round(p, 2),
                              "prediction": "diabetic" if p >= 0.5 else "not diabetic"},
            "memory_based": memory_predict(glucose, bmi, age)}


agent = Agent(
    name="Diabetes Risk Agent",
    model="gpt-4o-mini",
    tools=[assess_risk],
    instructions=(
        "You explain diabetes risk. If glucose, BMI, blood pressure, or age is 0 or impossible, "
        "ask the user for a valid value and do not call the tool. Otherwise call assess_risk. "
        "Explain both results side by side in plain words. If the memory-based confidence is low or none, say so. "
        "If the two models disagree, say that clearly. End with: 'This is not medical advice. Please see a doctor.'"
    ),
)

## 5. Tests

In [ ]:
tests = {
    "D1 high risk": "Pregnancies 6, glucose 148, blood pressure 72, skin 35, insulin 0, BMI 33.6, pedigree 0.627, age 50",
    "D2 low risk": "Pregnancies 1, glucose 85, blood pressure 66, skin 29, insulin 0, BMI 26.6, pedigree 0.351, age 31",
    "D3 no similar cases": "Pregnancies 2, glucose 250, blood pressure 90, skin 40, insulin 300, BMI 60, pedigree 1.2, age 90",
    "D4 invalid BMI": "Pregnancies 1, glucose 120, blood pressure 70, skin 20, insulin 80, BMI 0, pedigree 0.5, age 35",
}

for name, text in tests.items():
    result = await Runner.run(agent, text)
    print(f"=== {name} ===\n{result.final_output}\n")